# IMPORTS

In [5]:
from IPython.display import HTML, display
display(HTML('<style>.container { width:90% !important; }</style>'))
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
import pandas as pd
import openpyxl
from tkinter import simpledialog
import re
import shutil
import os
import requests
import json
from geopy.geocoders import Nominatim
from datetime import timedelta, datetime
from docxtpl import DocxTemplate
import subprocess
import pickle
import os.path
import secrets
from randomtimestamp import randomtimestamp, random_time
from random import randint
from docx import Document
from PyPDF2 import PdfMerger
from IPython.display import FileLink, FileLinks
from time import sleep
from geopy.exc import GeocoderTimedOut
import locale

    pd.get_option('display.max_rows')
    pd.get_option('display.max_columns')
    pd.get_option('display.max_colwidth')
    pd.get_option('display.expand_frame_repr')
    pd.set_option('display.expand_frame_repr', False)

# CONSTANTES

    june = [
        "28/06/23", "29/06/23", "30/06/23"
    ]
    
    july = [
        "03/07/23", "04/07/23", "05/07/23", "06/07/23", "07/07/23",
        "10/07/23", "11/07/23", "12/07/23", "13/07/23",
        "17/07/23", "18/07/23", "19/07/23", "20/07/23", "21/07/23",
        "24/07/23", "25/07/23", "26/07/23", "27/07/23", "28/07/23"
    ]
    types = ["itinerary", "call", "park", "poi", "break"]
    
    agenda = {}
    for date in june + july:
        agenda[date] = {}
    
    with open("/home/julien/BioCodex/agenda.json", "w") as file:
        json.dump(agenda, file, default=str)
        file.close()

In [6]:
locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8');
geolocator = Nominatim(user_agent="myGeocoder")

filepath="/home/julien/BioCodex/carnet_de_bord.xlsx"
filepath2="/home/julien/BioCodex/carnet_de_bord.xls"

durations = [0.75, 1, 1.25, 1.5]

zones = pd.read_excel(filepath, sheet_name="zones")
zones.set_index("cp", inplace=True)

Error: unsupported locale setting

# GEOCODE

In [3]:
def do_geocode(address, attempt=1, max_attempts=5):
    try:
        return geolocator.geocode(address)
    except GeocoderTimedOut:
        if attempt <= max_attempts:
            return do_geocode(address, attempt=attempt+1)
        raise

# DOCX TO PDF

In [4]:
def pdf_from_docx(input_docx, output_pathname):

    subprocess.call(
        [
            'soffice',
            # '--headless',
            '--convert-to',
            'pdf',
            '--outdir',
            output_pathname,
            input_docx
        ],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

    return None

# GET DAY

In [6]:
def get_day(date):
    
    home = {
        "type": "poi",
        "ddv": f"{date} 07:30",
        "nom": "HOME",
        "adresse": "35 RUE MARCEL BONTEMPS",
        "cp": 92100,
        "ville": "BOULOGNE BILLANCOURT"  
    }
    lunch = {
        "type": "break",
        "ddv": f"{date} 13:30",
        "nom": "LUNCH HOME",
        "adresse": "35 RUE MARCEL BONTEMPS",
        "cp": 92100,
        "ville": "BOULOGNE BILLANCOURT"  
    }
    diner = {
        "type": "poi",
        "ddv": f"{date} 19:30",
        "nom": "BACK HOME",
        "adresse": "35 RUE MARCEL BONTEMPS",
        "cp": 92100,
        "ville": "BOULOGNE BILLANCOURT"  
    }

    df_h = pd.DataFrame([home, lunch, diner])
    
    df_h["lat"] = pd.Series(dtype=float, index=df_h.index)
    df_h["lon"] = pd.Series(dtype=float, index=df_h.index)
    
    df_h.columns=[
        "type", "ddv", "id", "adr", "zip", "city", "lat", "lon" 
    ]
    
    for i, row in df_h.iterrows():
        sleep(1)
        loc = do_geocode(f'{row["adr"]} {str(row["zip"])} {row["city"]}')
        sleep(1)
        df_h.loc[i, "lat"] = loc.latitude
        df_h.loc[i, "lon"] = loc.longitude
        
    cdb = pd.read_excel(filepath2, sheet_name="pds")
    cdb["ddv"] = cdb["ddv"].astype(str)
   
    ts = pd.to_datetime(date, format="%d/%m/%y")
    match = ts.date().strftime("%Y-%m-%d")
        
    tournee = cdb[["ddv", "nom", "adresse", "cp", "ville"]][cdb["ddv"].str.contains(match)].copy()
    tournee.columns = ["ddv", "id", "adr", "zip", "city"]
    sleep(1)
    tournee["loc"] = [do_geocode(f'{tournee.loc[i, "adr"]} {int(tournee.loc[i, "zip"])} {tournee.loc[i, "city"]}') for i in tournee.index]
    sleep(1)
    tournee["lat"] = [tournee.loc[i, "loc"].latitude for i in tournee.index]
    tournee["lon"] = [tournee.loc[i, "loc"].longitude for i in tournee.index]
    tournee["type"] = "call"
    
    del tournee["loc"]
    
    day = pd.concat([tournee, df_h], ignore_index=True)
    day["ddv"] = pd.to_datetime(day["ddv"])
    day = day.sort_values(by=["ddv"]).fillna("")
    day.reset_index(drop=True, inplace=True)
    day["type"] = day["type"].replace("", "call")
    day["zip"] = day["zip"].astype(int)
    
    return day

# RANDOM DEPARTURE

In [7]:
def get_random_time(date, h_min="08:30", h_max="09:30"):

    t_min = datetime.strptime(f"{h_min}", "%H:%M").time()
    t_max = datetime.strptime(f"{h_max}", "%H:%M").time()
    rand_t = random_time(start=t_min, end=t_max, pattern="%H:%M")
    rand_time = rand_t.strftime("%H:%M")

    return pd.to_datetime(f"{date} {rand_time}", format="%d/%m/%y %H:%M")

# ITINARIES & CAR PARK

    datetime.strptime(date, "%d/%m/%y").date().strftime("%d %B %Y")

In [8]:
def get_itinaries_and_parks(date: str, day: pd.DataFrame):
    
    departure_d_t = get_random_time(date)
    print(f"day begins at {departure_d_t.time()}")
    
    ts = pd.to_datetime(date, format="%d/%m/%y")
    filename = ts.date().strftime("%Y%m%d")
    
    trajets = pd.DataFrame(columns = ["from", "to", "distance", "departure", "duration", "arrival"])
    parkings = pd.DataFrame(columns=["pbp_date", "park", "start", "stay", "stop", "code_zone", "nom_zone", "ville", "price"])
    
    for i in range(day.shape[0]-1):
        
        origin = day.loc[i]
        lat_0 = day.loc[i, "lat"]
        lon_0 = day.loc[i, "lon"]    
        trajets.loc[i, "from"] = origin["id"]

        destination = day.loc[i+1]
        lat_1 = day.loc[i+1, "lat"]
        lon_1 = day.loc[i+1, "lon"]
        trajets.loc[i, "to"] = destination["id"]

        trajets.loc[i, "departure"] = departure_d_t.time()
        # print(f"route departure: {departure_d_t.time()}")

        r = requests.get(f"http://router.project-osrm.org/route/v1/car/{lon_0},{lat_0};{lon_1},{lat_1}?overview=false""")
        sleep(1)

        routes = json.loads(r.content)
        route = routes.get("routes")[0]

        duration = int(round(route["duration"]/60,0))
        trajets.loc[i, "duration"] = duration
        # print(f"route duration: {duration} minutes")
        
        arrival_d_t = datetime.combine(departure_d_t.date(), departure_d_t.time()) + timedelta(minutes = duration)
        trajets.loc[i, "arrival"] = arrival_d_t.time()
        # print(f"route end: {arrival_d_t.time()}")
        
        distance = round(route['distance']/1000, 1)    
        trajets.loc[i, "distance"] = distance
        
        
        if destination["type"] == "call":            
            
            start = arrival_d_t + timedelta(minutes=9)
            # print(f"parking start: {start.time()}")
            
            parkings.loc[i, "pbp_date"] = datetime.strptime(date, "%d/%m/%y").date().strftime("%d %B %Y")
            parkings.loc[i, "park"] = f'PARK_{destination["id"]}'            
            parkings.loc[i, "start"] = start.time().strftime("%H:%M")
            parkings.loc[i, "stay"] = durations[randint(0,3)]
            # print(f'parking duration: {parkings.loc[i, "stay"]*60} minutes')
            
            stop = start + timedelta(minutes=parkings.loc[i, "stay"]*60)
            # print(f"parking end: {stop.time()}")
            
            parkings.loc[i, "stop"] = stop.time().strftime("%H:%M")
            code_zone = destination["zip"]
            parkings.loc[i, "code_zone"] = code_zone
            parkings.loc[i, "nom_zone"] = zones.loc[code_zone, "nom zone"]
            parkings.loc[i, "pbp_ville"] = zones.loc[code_zone, "ville"]
            price = (parkings.loc[i, "stay"] * zones.loc[code_zone, "tarif"]) + 0.2
            parkings.loc[i, "price"] = price
            
            departure_d_t = stop + timedelta(minutes=8)
            
            
        elif destination["type"] == "break":
            
            start = arrival_d_t + timedelta(minutes=9)
            # print(f"break start: {start.time()}")
            
            break_duration = 45
            # print(f"break duration: {break_duration} minutes")
            
            stop = start + timedelta(minutes=break_duration)
            # print(f"break stop: {stop.time()}")
            
            departure_d_t = stop + timedelta(minutes=8)
            
        
        # elif destination["type"] == "poi":
            
            # print(f"end of the day: {arrival_d_t.time()}")
            
            
    return trajets, parkings

# GET INVOICES

In [9]:
def get_pbp_invoices(date: str):
    
    day = get_day(date)      
    trajets, parkings = get_itinaries_and_parks(date, day)
    ts = pd.to_datetime(date, format="%d/%m/%y")
    filename = ts.date().strftime("%Y%m%d")
    
    with pd.ExcelWriter(f"/home/julien/BioCodex/Tournées/{filename}.xlsx") as writer:
        
        day.to_excel(writer, sheet_name="day", index=False)
        trajets.to_excel(writer, sheet_name="trajets", index=False)
        parkings.to_excel(writer, sheet_name="parkings", index=False)
    
    prix_total = "{:.2f}".format(parkings["price"].sum())
    dist_total = int(round(trajets["distance"].sum(), 0))    
    month = ts.date().strftime("%B")
    
    common_path = f"./frais/{month}"

    pages = []
    
    for i, parking in parkings.iterrows():
        
        park = {
            "date": parking["pbp_date"],
            "start": parking["start"],
            "end": parking["stop"],
            "ville": parking["pbp_ville"],
            "nom": parking["nom_zone"],
            "code": parking["code_zone"],
            "price": "{:.2f}".format(parking["price"])
        }
        
        page = DocxTemplate("/home/julien/BioCodex/Frais/pbp_tpl.docx")
        page.render(park)
        path_docx = f"{common_path}/docx/pbp_{filename}_{i+1}.docx"
        page.save(path_docx)                
        pdf_from_docx(path_docx, f"{common_path}/pdfs/")
        os.remove(path_docx)
        pages.append(f"{common_path}/pdfs/pbp_{filename}_{i+1}.pdf")
    

    merger = PdfMerger()

    for page in pages:
        merger.append(page)
    
    merger.write(f"{common_path}/pdfs/pbp_{filename}.pdf")
    merger.close()

    for page in pages:
        os.remove(page)
    
        
    return prix_total, dist_total

  # EXECUTE

In [13]:
execute = {}

In [14]:
def process_date(date):
    
    print(date)
    
    prix_total, dist_total = get_pbp_invoices(date)
    
    print(f"kms:\t\t{dist_total}\nparkings:\t{prix_total}\n")

    date = datetime.strptime(date, "%d/%m/%y").date().strftime("%Y%m%d")
    
    execute[date] = {
        "total parkings" : prix_total,
        "total kms": dist_total,
        "filepath": f"./frais/juillet/pdfs/pbp_{date}.pdf"
    }
    
    return execute

def make_clickable(url):
    
    name= os.path.basename(url)
    
    return '<a href="{}">{}</a>'.format(url,name)

    for date in june:
        execute = process_date(date)

In [15]:
for date in july:
    execute = process_date(date)

03/07/23
day begins at 09:27:00
kms:		13
parkings:	18.80

04/07/23
day begins at 08:30:00
kms:		24
parkings:	19.80

05/07/23
day begins at 09:20:00
kms:		21
parkings:	19.80

06/07/23
day begins at 09:23:00
kms:		9
parkings:	11.60

07/07/23
day begins at 09:25:00
kms:		34
parkings:	20.00

10/07/23
day begins at 08:43:00
kms:		11
parkings:	17.60

11/07/23
day begins at 08:52:00
kms:		10
parkings:	22.80

12/07/23
day begins at 08:37:00
kms:		12
parkings:	14.60

13/07/23
day begins at 08:30:00
kms:		27
parkings:	23.80

17/07/23
day begins at 08:46:00
kms:		25
parkings:	15.80

18/07/23
day begins at 09:06:00
kms:		35
parkings:	21.80

19/07/23
day begins at 09:08:00
kms:		43
parkings:	21.80

20/07/23
day begins at 08:45:00
kms:		16
parkings:	13.60

21/07/23
day begins at 08:36:00
kms:		31
parkings:	17.10

24/07/23
day begins at 08:49:00
kms:		36
parkings:	12.60

25/07/23
day begins at 09:06:00
kms:		10
parkings:	9.40

26/07/23
day begins at 09:00:00
kms:		24
parkings:	23.10

27/07/23
day beg

In [ ]:
resume = pd.DataFrame.from_dict(execute, orient="index")

In [18]:
resume.index = pd.to_datetime(resume.index, format="%Y%m%d").strftime('%d/%m/%y')

In [20]:
resume.style.format({'filepath': make_clickable})

,total parkings,total kms,filepath
03/07/23,18.80,13,pbp_20230703.pdf
04/07/23,19.80,24,pbp_20230704.pdf
05/07/23,19.80,21,pbp_20230705.pdf
06/07/23,11.60,9,pbp_20230706.pdf
07/07/23,20.00,34,pbp_20230707.pdf
10/07/23,17.60,11,pbp_20230710.pdf
11/07/23,22.80,10,pbp_20230711.pdf
12/07/23,14.60,12,pbp_20230712.pdf
13/07/23,23.80,27,pbp_20230713.pdf
17/07/23,15.80,25,pbp_20230717.pdf


# UPDATE XLSX BOOK

In [ ]:
def update_calls_sheet(date, filepath=filepath):
    
    calls_of_the_day = get_calls_by_date(date)
    print(f"nb of calls: {calls_of_the_day.shape[0]-3}")
        
    with pd.ExcelWriter(
        filepath, engine='openpyxl', if_sheet_exists="replace", mode="a", date_format="dd/mm/yy", datetime_format="HH:MM"
    ) as writer:

        if "calls" not in writer.sheets:
            
            calls_of_the_day.to_excel(writer, sheet_name="calls", index=False)  
            print("Creating sheet 'calls'")
            # print("Calls: sheet updated")
            
        else:
            
            existing_calls = pd.read_excel(filepath, sheet_name="calls", engine='openpyxl')

            dates = set([row["ddv"].date() for i, row in existing_calls.iterrows()])
            
            ts = pd.to_datetime(date, format="%d/%m/%y")

            if ts.date() in dates:

                print("date already in calls")
                # print("no need to update sheet calls")

            else:

                concat = pd.concat([existing_calls, calls_of_the_day])
                
                concat.to_excel(writer, sheet_name="calls", index=False)
                
                # print("Calls: sheet updated")

    return None 

In [ ]:
def update_kms_sheet(day_steps, filepath=filepath):

    date = day_steps[0]
    dist_tot_park_tot_dict = {}
    dist_tot_park_tot_dict[date] = {}
    dist_tot_park_tot_dict[date]["kms"] = day_steps[-2]
    dist_tot_park_tot_dict[date]["parkings"] = day_steps[-1]
    
    kms_of_the_day = pd.DataFrame.from_dict(dist_tot_park_tot_dict, orient="index")
    kms_of_the_day.reset_index(drop=False, inplace=True)
    kms_of_the_day.columns=["date", "kms pro", "total parking"]
            
    with pd.ExcelWriter(
        filepath, engine='openpyxl', if_sheet_exists="replace", mode="a", date_format="dd/mm/yy", datetime_format="HH:MM"
    ) as writer:    
        
        if "kms" not in writer.sheets:
            
            kms_of_the_day.to_excel(writer, sheet_name="kms", index=False)  
            print("Creating sheet 'kms'")
            # print("Kms: sheet updated")
            
        else:

            kmsdf = pd.read_excel(filepath, sheet_name="kms", engine='openpyxl')

            dates = set([row["date"] for i, row in kmsdf.iterrows()])

            if date in dates:

                print("date already in kms")
                # print("no need to update sheet kms")

            else:
                
                concat = pd.concat([kmsdf, kms_of_the_day])
                concat.to_excel(writer, sheet_name="kms", index=False)

                # print("Kilometers: sheet updated")
                                
    return None

In [ ]:
day_steps = get_day_steps("28/06/23")

In [ ]:
day_steps[-1]

In [ ]:
def update_sheets(date, filepath=filepath):

    update_calls_sheet(date)
    update_kms_sheet(date)

    return None